# Set up your own OGC API

In this notebook you'll publish a GeoPackage as a standards-compliant **OGC API – Features** service, using Kartverket's [OGC API starter kit](https://github.com/kartverket/ogc-api-starter) and the same software Kartverket runs in production:

- **backend**: pygeoapi, the API itself (`backend/Dockerfile`)
- **frontend**: Kartverket's web pages on top of the API (`frontend/Dockerfile`)

```
data/demo.gpkg ──(optional: PostGIS)──▶ backend (pygeoapi) ──▶ frontend ──▶ your browser
                                               ▲                    ▲
                                               └── config/pygeoapi-config.yml (this notebook writes it)
```

Run the cells from top to bottom: click ▶ on each cell, or press **Shift+Enter**. **Run All** at the top works too. The cells you're meant to edit are marked ✏️.

> The first time you run a cell, VS Code may ask you to pick a kernel. Choose **Python Environments → Python 3.12** (`/usr/local/bin/python`).

In [ ]:
import json
from workshop import *

## 1. Download the software

Downloads Kartverket's backend and frontend images and builds the starter kit on top of them.

In [ ]:
download()

## 2. Your data ✏️

The workshop ships with a small placeholder dataset, `data/demo.gpkg` (Norwegian towns, stored in EPSG:25833).

**Want to use your own data?** Drag a GeoPackage (`.gpkg`) from your computer into the `data/` folder in the Explorer on the left, change `GPKG` below and re-run the cell. Every layer in the GeoPackage becomes a *collection* in the API.

**Where should the API read the data from?** Pick a `SOURCE`:

| `SOURCE` | |
|---|---|
| `"gpkg"` | Straight from the GeoPackage file. Simplest, with no database. |
| `"postgis"` | From a PostGIS database. **This is what Kartverket recommends for real services**, and it adds two things: vector tiles (OGC API – Tiles) and downloads (GeoPackage/CSV) in the frontend. You don't need a database of your own: the cell below starts one here in the Codespace and copies your GeoPackage into it. |

In [ ]:
GPKG = "data/demo.gpkg"
SOURCE = "gpkg"   # or "postgis"

layers = prepare_data(GPKG, SOURCE)

## 3. Describe the dataset ✏️

This is the metadata for the service as a whole. It's shown on the landing page and in the API's own description of itself.

In [ ]:
DATASET = {
    "title": "Byer i Norge",
    "description": "Et lite eksempeldatasett laget for OGC API-workshopen.",
    "keywords": ["by", "tettsted", "workshop"],
    "provider": {
        "name": "Din organisasjon",
        "url": "https://www.example.com",
    },
    "contact": {
        "name": "Ditt navn",
        "email": "deg@example.com",
        "url": "https://www.example.com",
    },
    "license": {
        "name": "CC BY 4.0",
        "url": "https://creativecommons.org/licenses/by/4.0/",
    },
}

## 4. Describe each collection ✏️

One entry per layer in your GeoPackage, keyed by **layer name** (as listed in step 2).

- `id`: the collection's name in the URL: `/collections/<id>`. Kartverket's [URL standard](https://kartverket.github.io/ogcapi-docs/docs/url-standard-ogc-api) says: lowercase, words separated by hyphens, `æ/ø/å` written as `ae/o/a`, and preferably a plural noun (`kommuner`, not `kommune`).
- `id_field`: the column that uniquely identifies a feature. It becomes the feature's URL: `/collections/<id>/items/<id_field value>`.
- `title_field` (optional): a human-readable column, used as the feature's name in the frontend.

Switched to your own GeoPackage? Run step 5 once anyway: it prints a ready-to-paste entry for every layer that's missing here.

In [ ]:
COLLECTIONS = {
    "byer": {
        "id": "byer",
        "title": "Byer",
        "description": "Et utvalg norske byer med kommune og fylke.",
        "keywords": ["by", "kommune", "fylke"],
        "id_field": "fid",
        "title_field": "navn",
    },
}

## 5. Generate the configuration

Turns your metadata into `config/pygeoapi-config.yml`. It's the same file as in the starter kit, and both the backend and the frontend are built with it. Have a look at what came out, and compare it with the [starter kit's example](https://github.com/kartverket/ogc-api-starter/blob/main/config/pygeoapi-config.yml). With `SOURCE = "postgis"` the `providers` block looks just like the starter kit's.

In [ ]:
config = build_config(DATASET, COLLECTIONS, layers, SOURCE)
write_config(config)

print(CONFIG_FILE.read_text(encoding="utf-8"))

## 6. Start your API 🚀

In [ ]:
start()
show_links(layers)

Open the **landing page** link above. That's the Kartverket frontend, running on your API.

> The links only work for you while you're signed in to GitHub, because Codespaces ports are private by default. See step 8 to share them.

## 7. Use the API like a client would

An OGC API is just HTTP + JSON, so any tool that can fetch a URL can use it. A few examples:

In [ ]:
# Which collections are there?
for c in get_json("/collections")["collections"]:
    print(c["id"], "-", c["title"])

In [ ]:
# The first three features of the first collection, as GeoJSON
cid = layers[0]["id"]
items = get_json(f"/collections/{cid}/items?limit=3")
print(json.dumps(items["features"], indent=2, ensure_ascii=False)[:1500])

In [ ]:
# Filter by bounding box (lon/lat), and ask for coordinates in EPSG:25833 instead of WGS84
items = get_json(
    f"/collections/{cid}/items?bbox=4,58,12,62"
    "&crs=http://www.opengis.net/def/crs/EPSG/0/25833"
)
print(len(items["features"]), "features in the bbox")
for f in items["features"][:5]:
    print(f["id"], f["geometry"]["coordinates"])

**Try it yourself:** edit the URLs above. Some things to try:

- `?limit=5&offset=10`: paging
- `?<column>=<value>`, e.g. `?fylke=Innlandet`: filter on an attribute
- `/collections/<collection>/items/<id>`: a single feature
- `/collections/<collection>/queryables`: which attributes you can filter on
- `/collections/<collection>/tiles`: the vector tiles (with `SOURCE = "postgis"`). The frontend has a tiles page for each collection too.

## 8. Open it in QGIS or ArcGIS (optional)

First, open the **Ports** tab at the bottom of VS Code, right-click port **3000** → **Port Visibility → Public**. Anyone with the link can now reach your API, so only do this with data you're happy to share. Then use the **landing page** URL from step 6. It's the same address for people, QGIS and scripts:

- **QGIS:** **Layer → Add Layer → Add WFS / OGC API – Features Layer… → New**. Paste the URL, set **Version** to *OGC API – Features*, then **Connect** to list your collections.
- **ArcGIS Pro:** **Insert → Connections → Server → New OGC API Server**, and paste the URL.

More in Kartverket's guide: [Bruk i GIS-verktøy](https://kartverket.github.io/ogcapi-docs/docs/bruk).

## 9. Change something

Edit the metadata in step 3 or 4, then re-run **steps 5 and 6**. Changed the GeoPackage or `SOURCE`? Re-run from **step 2**. Reload the page in your browser to see the change.

## Stop

Stops the API (and the database, if you used one). Your files stay as they are; run step 6 again to start it back up.

To stop, remove the `#` and run the cell. It's commented out so that **Run All** doesn't stop your API straight after starting it. If your Codespace goes to sleep, the API comes back by itself when you return.

In [ ]:
# stop()